# 02 · Target definition, features and validation design (Phases 5–6)

This notebook fixes **what is predicted, from which inputs, and how performance will be measured** — before any
model is trained, so the evaluation cannot be tuned to flatter a model. No model is fitted here.

Code: `src/flameguard/dataset.py` (Phase 5), `features.py` and `validation.py` (Phase 6).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression

from flameguard.dataset import build_modeling_data, VARIANTS
from flameguard.features import FlexFeatures, make_preprocessor, applicability_domain, monotone_constraints
from flameguard.validation import repeated_group_kfold, stress_splits, fold_table, check_splits

master = pd.read_csv(ROOT / "data" / "processed" / "combustion_master.csv")
pd.set_option("display.width", 160)

## Phase 5 · Target

**Prediction target `y_sustained`** — did the burning droplet keep burning instead of self-extinguishing while fuel
remained?

| NASA `Test end` | y_sustained | Reasoning |
|---|---|---|
| Extinction | 0 | flame went out at a finite droplet size (fuel left) |
| Completion | 1 | burned until the droplet was consumed |
| Disruption | 1 | still burning when the droplet broke up; extinction had not occurred |

Treating Disruption as sustained is a judgement call, so the `exclude_disruption` variant re-runs everything without
those tests. If a conclusion only holds in one variant, we report that.

**Fire Risk Score (definition fixed now, values come from Phase 9).** `Fire Risk = 100 × P(sustained)`, using
*calibrated* cross-validated probabilities. It means: "estimated chance that a burning fuel droplet in this
atmosphere keeps burning rather than self-extinguishing, within FLEX test conditions". Risk-level bands (Low / High …)
will be set from the validated operating threshold in Phase 9, not picked arbitrarily now.

In [2]:
rows = []
for v in VARIANTS:
    s = build_modeling_data(master, v).summary()
    rows.append({k: s[k] for k in ["variant", "tests", "sustained", "extinguished", "groups"]} | {"excluded": s["excluded"]})
pd.DataFrame(rows)

,variant,tests,sustained,extinguished,groups,excluded
0,primary,252,80,172,44,"{'missing_d0': 12, 'out_of_scope_pressure_2-3a..."
1,exclude_disruption,199,27,172,43,"{'disruption_excluded_by_variant': 53, 'missin..."
2,impute_d0,264,86,178,44,"{'out_of_scope_pressure_2-3atm': 9, 'missing_p..."


### Scope and exclusions (primary variant)
Every test not used is listed with its reason — nothing disappears silently.

In [3]:
data = build_modeling_data(master, "primary")
display(data.excluded["reason"].value_counts().rename("tests"))
data.excluded

reason
missing_d0                      12
out_of_scope_pressure_2-3atm     9
missing_pressure                 1
Name: tests, dtype: int64

,test_id,flex_identifier,fuel,outcome_raw,reason
0,9,30CAL07,Heptane,Disruption,missing_d0
1,10,256F002,Methanol,Extinction,out_of_scope_pressure_2-3atm
2,11,256R001,Methanol,Extinction,out_of_scope_pressure_2-3atm
3,12,264F003,Heptane,Disruption,out_of_scope_pressure_2-3atm
4,13,256R002,Methanol,Extinction,out_of_scope_pressure_2-3atm
5,14,256F007,Methanol,Extinction,out_of_scope_pressure_2-3atm
6,15,256F008,Methanol,Extinction,out_of_scope_pressure_2-3atm
7,16,256F009,Methanol,Extinction,out_of_scope_pressure_2-3atm
8,17,240R003,Methanol,Disruption,out_of_scope_pressure_2-3atm
9,18,248F001,Heptane,Extinction,out_of_scope_pressure_2-3atm


- **2–3 atm (9 tests):** a single CO₂ series; pressure and CO₂ cannot be separated there.
- **Missing d₀ (12) / pressure (1):** no imputation in the primary model; `impute_d0` is the sensitivity check.
- Flagged-but-valid tests (e.g. 70 and 73, fuel-dispensing anomalies) are **kept**; their inputs and outcomes were
  measured. Phase 9 checks whether removing them changes anything.

In [4]:
pd.crosstab([data.X["fuel"], data.meta["diluent"]], data.y.map({0: "extinguished", 1: "sustained"}), margins=True)

y_sustained       extinguished  sustained  All
fuel     diluent                              
Heptane  CO2                26         18   44
         He                 14          7   21
         N2                 16         25   41
Methanol CO2                54         13   67
         He                 24          2   26
         N2                 38         15   53
All                        172         80  252

Note the **methanol + He** cell: only 2 sustained tests. Any He-specific statement for methanol rests on
almost nothing, and the dashboard will say so.

## Phase 6 · Features

| Feature | Linear model | Tree models | Why |
|---|---|---|---|
| `is_heptane` | ✓ | ✓ | fuel is a strong effect (EDA) |
| `x_o2` | ✓ | ✓ | dominant signal; monotone-increasing constraint in XGBoost |
| `x_co2`, `x_he` | ✓ | ✓ | suppressant amounts |
| `pressure_atm` | ✓ | ✓ | test condition (0.7 / 1 atm) |
| `d0_mm` | ✓ | ✓ | droplet size |
| `heptane_x_d0` | ✓ | – | size effect has opposite sign per fuel; trees learn this themselves |
| `x_n2` | ✗ | ✗ | redundant: mole fractions sum to 1 |
| `p_o2_atm` | ablation | ablation | = x_o2 × pressure; tested as `with_po2` feature set |

Scaling, centring and imputation are fitted **inside each training fold** (sklearn `Pipeline`).

In [5]:
Xf = FlexFeatures(feature_set="base", interaction=True).fit_transform(data.X)
display(Xf.describe().T.round(3))
print("XGBoost monotone constraints (base set):", monotone_constraints(FlexFeatures().fit(data.X).feature_names_out_))

,count,mean,std,min,25%,50%,75%,max
is_heptane,252.0,0.421,0.495,0.000,0.000,0.000,1.000,1.000
x_o2,252.0,0.201,0.048,0.120,0.170,0.200,0.230,0.340
x_co2,252.0,0.103,0.141,0.000,0.000,0.000,0.200,0.500
x_he,252.0,0.050,0.115,0.000,0.000,0.000,0.000,0.450
pressure_atm,252.0,0.884,0.148,0.692,0.705,0.999,1.006,1.019
d0_mm,252.0,2.999,0.693,1.100,2.570,3.015,3.522,4.670
heptane_x_d0,252.0,-0.047,0.497,-1.899,0.000,-0.000,0.000,1.671


XGBoost monotone constraints (base set): (0, 1, 0, 0, 0, 0)


### Collinearity check (variance inflation factor; > 10 = severe)

In [6]:
def vif(frame):
    out = {}
    for col in frame.columns:
        others = frame.drop(columns=col)
        r2 = LinearRegression().fit(others, frame[col]).score(others, frame[col])
        out[col] = np.inf if r2 >= 1 else 1 / (1 - r2)
    return pd.Series(out).round(1)

with_n2 = Xf.drop(columns="heptane_x_d0").assign(x_n2=master.set_index("test_id").loc[data.meta["test_id"], "x_n2"].to_numpy())
pd.DataFrame({
    "chosen base set": vif(Xf.drop(columns="heptane_x_d0")),
    "+ x_n2": vif(with_n2),
    "+ p_o2_atm": vif(FlexFeatures(feature_set="with_po2").fit_transform(data.X)),
})

,chosen base set,+ x_n2,+ p_o2_atm
d0_mm,1.2,1.2,1.2
is_heptane,1.1,1.1,1.1
p_o2_atm,NaN,NaN,49.8
pressure_atm,2.3,2.6,35.7
x_co2,1.9,1865.1,2.5
x_he,1.7,1236.0,2.0
x_n2,NaN,2089.3,NaN
x_o2,1.8,207.0,58.1


**Read-out.** The chosen set has low VIFs. Adding `x_n2` makes the design nearly singular (mole fractions sum
to ~1), and adding `p_o2_atm` inflates `x_o2`/`pressure_atm` — confirming why both stay out of the main models.
VIF only measures *linear* redundancy across the whole dataset; the O₂–suppressant coupling *within* each test series
(EDA finding 4) is still there and is handled at interpretation time.

## Phase 6 · Validation design

**Primary:** 5 × repeated 5-fold `StratifiedGroupKFold`, grouped by `group_id_atmosphere`.
All tests that shared an atmosphere go to the same fold; the 25 splits are checked automatically for group leakage,
overlap, single-class folds, and that each repeat partitions the data exactly once.

In [7]:
splits = repeated_group_kfold(data.y, data.groups, n_splits=5, n_repeats=5, seed=0)
ft = fold_table(splits, data.y, data.groups)
display(ft.head(5))
ft[["train_tests", "test_tests", "test_groups", "test_sustained_share"]].describe().loc[["min", "mean", "max"]].round(2)

,split,train_tests,test_tests,test_groups,test_sustained,test_sustained_share
0,r0f0,205,47,9,13,0.277
1,r0f1,207,45,10,15,0.333
2,r0f2,205,47,7,13,0.277
3,r0f3,183,69,10,25,0.362
4,r0f4,208,44,8,14,0.318


,train_tests,test_tests,test_groups,test_sustained_share
min,183.0,44.0,7.0,0.28
mean,201.6,50.4,8.8,0.31
max,208.0,69.0,11.0,0.39


Test folds range from about 44 to 69 tests (the 29-test cabin-air group always lands in one fold) with 28–39%
sustained. Per-fold metrics will therefore be noisy — results are reported as mean ± standard deviation over all 25
folds, plus pooled out-of-fold metrics, never as a single lucky split.

**Stress tests** — conditions entirely absent from training:

In [8]:
stress = stress_splits(data.meta)
check_splits(stress, data.y, data.groups)
fold_table(stress, data.y, data.groups)

,split,train_tests,test_tests,test_groups,test_sustained,test_sustained_share
0,leave_out_CO2,141,111,18,31,0.279
1,leave_out_He,205,47,15,9,0.191
2,train_1atm_test_0.7atm,151,101,18,38,0.376
3,train_0.7atm_test_1atm,101,151,26,42,0.278


- `leave_out_He`: the training data then contains **no helium at all**, so `x_he` is constant (0) in training
  and the model cannot learn any helium effect. This measures how badly predictions degrade for an untested
  suppressant — an honest warning for the dashboard, not a fair performance number.
- `leave_out_CO2` and `leave_out_He` also cross campaign periods (EDA finding 6).
- Pressure transfer checks whether 1 atm results carry over to 0.7 atm and vice versa.

### Safety-oriented evaluation plan (fixed before training)
- **Missed fire = false negative** (predict extinguished, actually sustained). Report recall on the sustained class
  and the false-negative rate alongside ROC-AUC, PR-AUC, F1, precision, accuracy.
- **Calibration** (Brier score, reliability curve), because the Fire Risk Score is a probability.
- **Operating threshold** chosen *inside* the training folds to reach recall ≥ 0.90 on sustained; report the
  precision this costs on the test folds.
- **Baselines** every model must beat: always-predict-majority, and O₂-only logistic regression.

## Applicability domain (per-fuel ranges used to bound dashboard inputs)

In [9]:
applicability_domain(data.X).pivot(index="feature", columns="fuel", values=["min", "max"]).round(3)

min              max         
fuel         Heptane Methanol Heptane Methanol
feature                                       
d0_mm          1.100    1.610   4.670    4.530
pressure_atm   0.695    0.692   1.015    1.019
x_co2          0.000    0.000   0.450    0.500
x_he           0.000    0.000   0.450    0.450
x_o2           0.130    0.120   0.340    0.340